# Lab: clustering 178 wines

Companion notebook of the lesson *U2 Clustering* — https://insper.github.io/ML/2026.2/classes/clustering/

178 wines from three cultivars, clustered without looking at the cultivar. The **fundamental lab** (steps 1 to 6) scales the data, chooses k, compares with shuffled data and profiles the clusters; the **advanced lab** (steps 7 to 9) tries DBSCAN and HDBSCAN, looks through PCA and only at the end audits every decision against the true cultivars. Runs in Colab or locally with scikit-learn, pandas, numpy and matplotlib; no download needed (the data ships with scikit-learn).

Every step starts with a **Predict**: answer it before running the cell. Each **Simulation** is followed by an empty cell for your experiment; the solutions are on the lesson page.

# Fundamental lab — grouping 178 wines without labels

## Step 1 — load, and seal the cultivar

**Predict.** The 13 measurements include an alcohol percentage and a proline concentration in milligrams per liter. Do you expect their standard deviations to be similar? Which column will dominate a Euclidean distance between two wines in raw units?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
df = wine.frame
cultivar = df.pop('target')                 # the true cultivar: sealed until step 9
print(df.shape, '· missing values:', int(df.isna().sum().sum()))
sd = df.std().sort_values()
print(sd.iloc[[0, 1, -2, -1]].round(3))
print(f"largest std / smallest std = {sd.iloc[-1] / sd.iloc[0]:.0f}")

## Step 2 — scale, and why the indices cannot decide it

**Predict.** k-means with \(k = 3\), on the raw columns and after three scalers. Which version will the silhouette, CH and DB prefer? Should that preference decide the scaling?

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

X = df.to_numpy()
scalers = {'none': None, 'StandardScaler': StandardScaler(), 'MinMaxScaler': MinMaxScaler(),
           'RobustScaler': RobustScaler()}
for name, scaler in scalers.items():
    M = X if scaler is None else scaler.fit_transform(X)
    lab = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(M)
    print(f"{name:15s} silhouette {silhouette_score(M, lab):.3f}   CH {calinski_harabasz_score(M, lab):6.1f}"
          f"   DB {davies_bouldin_score(M, lab):.3f}")

Z = StandardScaler().fit_transform(X)       # the choice, made by argument

**Simulation: the raw distance is proline.** Step 1 claimed that, in raw units, the distance between two wines is essentially their difference in proline. Test it without opening the seal: cluster the raw data with \(k = 3\), then only the `proline` column, then the 12 other columns, and compare the partitions with each other with the ARI (the ARI compares any two partitions; no cultivar is needed).

In [ ]:
# your simulation here

## Step 3 — choose k with the elbow and the silhouette

**Predict.** You know there are three cultivars but are not allowed to use it. Will the elbow and the three indices agree on one \(k\)? By a wide margin or a narrow one?

In [ ]:
rows = []
for k in range(2, 9):
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    rows.append({'k': k, 'inertia': m.inertia_, 'silhouette': silhouette_score(Z, m.labels_),
                 'CH': calinski_harabasz_score(Z, m.labels_), 'DB': davies_bouldin_score(Z, m.labels_)})
scores = pd.DataFrame(rows).set_index('k')
print(scores.round(3))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.2))
scores['inertia'].plot(ax=ax1, marker='o', title='inertia: look for the elbow')
scores['silhouette'].plot(ax=ax2, marker='o', title='silhouette: higher is better')
plt.tight_layout()
plt.show()

## Step 4 — the ruler: the same pipeline on shuffled data

**Predict.** Shuffling each column independently keeps every measurement's distribution and destroys the relations between them. Will k-means on the shuffled wines reach a silhouette near the real 0.285, or far below it?

In [ ]:
rng = np.random.default_rng(0)
Z_shuffled = Z.copy()
for j in range(Z_shuffled.shape[1]):
    rng.shuffle(Z_shuffled[:, j])           # destroys the relations between columns, keeps each column
for k in [2, 3, 4]:
    lab = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(Z_shuffled)
    print(k, round(silhouette_score(Z_shuffled, lab), 3), round(calinski_harabasz_score(Z_shuffled, lab), 1))

**Simulation: one shuffle is one draw.** Step 4 shuffled the columns once. Repeat the shuffle with 20 different seeds, with \(k = 3\): how much does the shuffled silhouette vary? Does any shuffle come close to the real 0.285?

In [ ]:
# your simulation here

## Step 5 — fit k = 3 and inspect each cluster

**Predict.** Will the three clusters be equally cohesive? What would a negative silhouette mean for one wine?

In [ ]:
from sklearn.metrics import silhouette_samples

km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Z)
labels = km.labels_
s = silhouette_samples(Z, labels)
for j in range(3):
    print(f"cluster {j}: {np.sum(labels == j):3d} wines · mean silhouette {s[labels == j].mean():.3f}"
          f" · negative silhouettes {np.sum(s[labels == j] < 0)}")

## Step 6 — profile the clusters in the original units

**Predict.** The centers live in standardized units, which nobody can read. How do you turn a cluster into a sentence a winemaker can check?

In [ ]:
cols = ['alcohol', 'flavanoids', 'color_intensity', 'proline', 'od280/od315_of_diluted_wines']
print(df.assign(cluster=labels).groupby('cluster')[cols].mean().round(2).to_string())

z_means = pd.DataFrame(Z, columns=df.columns).assign(cluster=labels).groupby('cluster').mean()
for j, row in z_means.iterrows():
    top = row[row.abs().sort_values(ascending=False).index[:3]]
    print(f"cluster {j}:", ', '.join(f"{c} {v:+.2f} sd" for c, v in top.items()))

**To close the fundamental part.** In one sentence each: (1) why could the three indices of step 2 not decide the scaling? (2) What does the silhouette of shuffled data add to the real one? (3) How would you describe cluster 2 to a winemaker, without saying "cluster 2"?

## Your turn (fundamental)

At home:

1. **Stability.** A real group should reappear when the data is resampled. For \(k = 2, \dots, 7\), cluster 30 random subsamples of 80% of the wines, and for every pair of subsamples compute the ARI between their labels on the wines they share. Which \(k\) gives the highest and least variable agreement? Compare the margin between \(k = 2\) and \(k = 3\) with the silhouette's 0.017.
2. **Fewer measurements.** Cluster using only `flavanoids`, `od280/od315_of_diluted_wines` and `proline` (standardized), then only the three measurements that separate the clusters least in step 6. Compare their ARIs with the cultivars (after step 9). Why is choosing variables with the clusters' own profiles circular, and how would you make it honest?

# Advanced lab — other algorithms, and the answer key

The fundamental lab took every decision without labels: standardize, \(k = 3\), a silhouette four times that of shuffled data, three groups with a name. This part tries the density methods on the same wines, looks at the clusters through a 2-D window and finally opens the seal to audit every decision. Continue in the same notebook and session: the steps below use `df`, `X`, `Z`, `scalers`, `labels` and `cultivar` from steps 1 to 6.

## Step 7 — try DBSCAN and HDBSCAN

**Predict.** In 13 standardized dimensions, do you expect dense regions separated by sparse ones? Will DBSCAN find three clusters for some `eps`?

In [ ]:
from sklearn.cluster import DBSCAN, HDBSCAN
from sklearn.neighbors import NearestNeighbors

kdist = NearestNeighbors(n_neighbors=5).fit(Z).kneighbors(Z)[0][:, -1]
print('distance to the 5th neighbor, percentiles 10/50/90:', np.percentile(kdist, [10, 50, 90]).round(2))
for eps in [1.8, 2.0, 2.2, 2.4, 2.6]:
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Z)
    print(f"DBSCAN eps {eps}: {lab.max() + 1} clusters, {np.sum(lab == -1):3d} noise")
hd = HDBSCAN(min_cluster_size=10).fit_predict(Z)
print(f"HDBSCAN min_cluster_size=10: {hd.max() + 1} clusters, {np.sum(hd == -1)} noise")

**Simulation: how big must a cluster be?** HDBSCAN's only parameter is `min_cluster_size`. Try 5, 10, 15, 20 and 30 on `Z`. Does any value find three clusters? What happens as it approaches the size of the groups k-means found in step 5 (51 to 65 wines)?

In [ ]:
# your simulation here

## Step 8 — look at the clusters with PCA

**Predict.** Two principal components keep only part of the variance. For the same partition, will the silhouette measured in 2-D be higher or lower than the one measured in 13-D?

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(Z)
A = pca.transform(Z)
print('variance kept by 2 components:', pca.explained_variance_ratio_.round(3),
      round(pca.explained_variance_ratio_.sum(), 3))

lab_2d = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(A)
print('k-means on the 2 components, silhouette measured in 2-D :', round(silhouette_score(A, lab_2d), 3))
print('the same partition, silhouette measured in 13-D         :', round(silhouette_score(Z, lab_2d), 3))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
ax1.scatter(A[:, 0], A[:, 1], c=labels, cmap='tab10', vmin=0, vmax=9, s=15)
ax1.set_title('k-means, k = 3 (fit in 13-D)')
ax2.scatter(A[:, 0], A[:, 1], c=np.where(hd == -1, 7, hd), cmap='tab10', vmin=0, vmax=9, s=15)
ax2.set_title('HDBSCAN (gray = noise)')
plt.show()

## Step 9 — open the seal: audit every decision

**Predict.** The seal opens. Which decision do you expect the answer key to contradict: the scaling, the \(k\) or the algorithm? Will the least cohesive cluster of step 5 be the most wrong?

In [ ]:
from sklearn.metrics import adjusted_rand_score

partitions = {
    'k-means k=3, standardized': labels,
    'k-means k=2, standardized': KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(Z),
    'k-means k=3, raw units':    KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X),
    'k-means k=3, on 2 PCs':     lab_2d,
    'DBSCAN eps=2.4':            DBSCAN(eps=2.4, min_samples=5).fit_predict(Z),
    'HDBSCAN':                   hd,
}
for name, lab in partitions.items():
    print(f"{name:27s} ARI {adjusted_rand_score(cultivar, lab):.3f}")
print(pd.crosstab(labels, cultivar, rownames=['cluster'], colnames=['cultivar']))

**Simulation: the four scalers, audited.** Step 2 chose to standardize by an argument about units, against the three indices. Now that the seal is open, give every option of step 2 its ARI with the cultivars. Does the silhouette rank the options the way the ARI does?

In [ ]:
# your simulation here

**To close the lesson.** In one sentence each: (1) why did DBSCAN never find three groups of wines? (2) Why does the same partition score 0.561 in 2-D and 0.283 in 13-D? (3) The least cohesive cluster turned out to be pure: what does a low silhouette say, then?

## Your turn (advanced)

At home, with more time:

1. **Gaussian mixtures.** Fit `GaussianMixture(n_components=3, covariance_type='tied', n_init=5, random_state=0)` on `Z` and compare its ARI with k-means'. Then choose the number of components and the `covariance_type` by the lowest `bic(Z)`: does the BIC choose the configuration with the best ARI?
2. **The gap statistic** (Tibshirani, Walther & Hastie, 2001). For each \(k\), compare \(\log\) of the inertia on the real data with its average over 20 datasets drawn uniformly in the same bounding box. The gap is the difference; choose the smallest \(k\) with \(\text{gap}(k) \ge \text{gap}(k+1) - s_{k+1}\). Which \(k\) does it choose?